# World-model training results

Run all cells from the repository root or `src/experiments/partial-visual-augmentation-simple/`.
The output path is resolved from the repository root. The notebook loads
`metrics.csv` files from the world-model output directory. Expected run paths are
`seed-N/vector/metrics.csv` and `seed-N/combined/metrics.csv`.

Set `DATA_DIR` to select a different world-model output folder.

The CSV step is the **training epoch**. Wall time is cumulative **training seconds**;
it excludes evaluation, logging, and checkpoint writing. Sparse evaluation metrics
are plotted only where recorded. No smoothing is applied.

Step averages use the available seeds at each recorded step. Time averages use
linear interpolation onto the union of recorded times within the common time
range of all seeds for that metric and modality, without extrapolation. Seeds
have equal weight. A single seed uses its recorded times directly.

Test MAE at horizons 0 and 1 appears in adjacent columns. Older runs with
only unnumbered test MAE use one test column. All panels share the horizontal axis.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Find the repository above the kernel's working directory.
ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file()
        and (p / "src/experiments/partial-visual-augmentation-simple").is_dir()
    ),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository.")
DATA_DIR = ROOT / "outputs/partial-visual-augmentation-simple/world-models"
print(f"Reading training results from {DATA_DIR}")
STYLES = {"vector": "-", "combined": ":"}
files = sorted(DATA_DIR.glob("seed-*/*/metrics.csv"))
if not files:
    raise FileNotFoundError(f"No metrics.csv files found under {DATA_DIR}")

# Load each run and attach its seed, modality, and recorded wall time.
runs = []
for path in files:
    seed, modality = path.parent.parent.name, path.parent.name
    if not seed.startswith("seed-") or modality not in STYLES:
        raise ValueError(f"Unexpected run path: {path}")
    frame = pd.read_csv(path)
    if frame.empty or not {"metric", "step", "value"}.issubset(frame.columns):
        raise ValueError(f"Expected nonempty metric,step,value CSV: {path}")
    frame["step"] = pd.to_numeric(frame["step"], errors="raise")
    frame["value"] = pd.to_numeric(frame["value"], errors="raise")
    if (
        frame["metric"].isna().any()
        or not np.isfinite(frame[["step", "value"]]).all().all()
    ):
        raise ValueError(f"Missing or nonfinite metric data: {path}")
    if frame.duplicated(["metric", "step"]).any():
        raise ValueError(f"Duplicate metric/step rows: {path}")
    times = frame.loc[frame.metric == "pretrain/wm/wall_time", ["step", "value"]]
    times = times.sort_values("step").rename(columns={"value": "wall_time"})
    if times.empty or (times.wall_time.diff().dropna() <= 0).any():
        raise ValueError(f"Missing or non-increasing training wall time: {path}")
    frame = frame.merge(times, on="step", how="left", validate="many_to_one")
    if frame.wall_time.isna().any():
        raise ValueError(f"Missing wall time for recorded steps: {path}")
    runs.append(frame.assign(seed=seed, modality=modality))

data = pd.concat(runs, ignore_index=True)
if data.duplicated(["seed", "modality", "metric", "step"]).any():
    raise ValueError("Multiple CSVs contain the same seed/modality/metric/step.")
metrics = sorted(data.metric.unique())
seeds = sorted(data.seed.unique())
seed_colors = {seed: plt.get_cmap("tab20")(i % 20) for i, seed in enumerate(seeds)}
print(f"Loaded {len(files)} CSVs, {len(seeds)} seeds, and {len(metrics)} metrics.")
display(
    data.groupby(["seed", "modality"]).agg(
        rows=("value", "size"), metrics=("metric", "nunique")
    )
)

In [ ]:
def mean_curve(frame, axis):
    """Average seed curves at recorded steps or shared training times.

    Args:
        frame: Rows for one metric and modality.
        axis: The step or wall_time column.

    Returns:
        Horizontal coordinates and the equally weighted seed mean.
    """
    if axis == "step":
        mean = frame.groupby("step").value.mean()
        return mean.index.to_numpy(), mean.to_numpy()

    curves = [run.sort_values(axis) for _, run in frame.groupby("seed")]
    start = max(run[axis].min() for run in curves)
    stop = min(run[axis].max() for run in curves)
    if start > stop:
        raise ValueError(f"Seeds have no common time range for {frame.metric.iloc[0]}")
    grid = np.unique(np.concatenate([run[axis].to_numpy() for run in curves]))
    grid = grid[(grid >= start) & (grid <= stop)]
    values = [np.interp(grid, run[axis], run.value) for run in curves]
    return grid, np.mean(values, axis=0)


def plot_metrics(axis, average):
    """Plot metric groups in columns with a shared horizontal axis.

    Args:
        axis: The step or wall_time column.
        average: Whether to average seeds or draw each seed separately.
    """
    # Arrange metrics down each column.
    available = set(data.metric)
    test_suffixes = ("0_test", "1_test")
    if not any(metric.endswith(("_0_test", "_1_test")) for metric in available):
        test_suffixes = ("test",)
    state_names = ("cartPos", "poleAngle")
    columns = [
        ["loss_total", "loss_rec", "loss_dyn"],
        *[
            [
                f"mae_{name}_{split}"
                for name in state_names
                if f"pretrain/wm/mae_{name}_{split}" in available
            ]
            for split in ("train", *test_suffixes)
        ],
        ["grad_steps", "wall_time"],
    ]
    rows = max(map(len, columns))
    fig, axes = plt.subplots(
        rows,
        len(columns),
        sharex=True,
        squeeze=False,
        figsize=(5 * len(columns), 3.3 * rows),
        layout="constrained",
    )
    positions = [
        (axes[row, col], f"pretrain/wm/{name}")
        for col, names in enumerate(columns)
        for row, name in enumerate(names)
    ]
    for ax, metric in positions:
        frame = data.loc[data.metric == metric]
        for modality, style in STYLES.items():
            subset = frame.loc[frame.modality == modality]
            if subset.empty:
                continue
            if average:
                x, y = mean_curve(subset, axis)
                ax.plot(
                    x,
                    y,
                    linestyle=style,
                    marker=".",
                    label=modality,
                    color={"vector": "tab:blue", "combined": "tab:orange"}[modality],
                )
            else:
                for seed, run in subset.groupby("seed"):
                    run = run.sort_values(axis)
                    ax.plot(
                        run[axis],
                        run.value,
                        linestyle=style,
                        marker=".",
                        color=seed_colors[seed],
                        label=f"{modality} / {seed}",
                    )
        ax.set_title(metric.removeprefix("pretrain/wm/"))
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize="small")
    # Label the last occupied axis in each column.
    xlabel = (
        "Training step (epoch)" if axis == "step" else "Training wall time (seconds)"
    )
    for col, names in enumerate(columns):
        for ax in axes[len(names) :, col]:
            ax.set_visible(False)
        if not names:
            continue
        bottom = axes[len(names) - 1, col]
        bottom.set_xlabel(xlabel)
        bottom.tick_params(axis="x", labelbottom=True)
    fig.suptitle("Mean across seeds" if average else "Individual seeds")
    plt.show()

## 1. All metrics against training step, averaged over seeds

In [ ]:
plot_metrics("step", average=True)

## 2. All metrics against wall time, averaged over seeds

In [ ]:
plot_metrics("wall_time", average=True)

## 3. All metrics against training step, one trace per seed

In [ ]:
plot_metrics("step", average=False)

## 4. All metrics against wall time, one trace per seed

In [ ]:
plot_metrics("wall_time", average=False)